# Meridian Financial — Fraud Model v2

Create the new fraud model Meridian wants to deploy.

v1 is intentionally **85% less sensitive to `distance_from_last_transaction`**.
v2 restores that fitted distance coefficient to its full baseline value, making the
new model more sensitive to fraud signals associated with transaction distance.

The serving contract, feature order, model name, and deployment flow remain unchanged.
v2 is written at `models/fraud/2/model.onnx`; v1 is not overwritten.


In [ ]:
from pathlib import Path
from copy import deepcopy
import os
import json
import numpy as np
import pandas as pd
import sklearn
import onnx

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType

print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("sklearn:", sklearn.__version__)
print("onnx:", onnx.__version__)

In [ ]:
FEATURES = [
    "distance_from_last_transaction",
    "ratio_to_median_purchase_price",
    "used_chip",
    "used_pin_number",
    "online_order",
]

DISTANCE_FEATURE = "distance_from_last_transaction"
DISTANCE_SENSITIVITY_MULTIPLIER = 0.15  # v1 keeps 15% of baseline sensitivity

def is_repo_root(path: Path) -> bool:
    return (
        (path / "data" / "train.csv").is_file()
        and (path / "data" / "validate.csv").is_file()
        and (path / "data" / "test.csv").is_file()
    )

# 1) Prefer the current working directory (normal when the notebook is opened from the repo).
cwd = Path.cwd().resolve()
candidates = []

if is_repo_root(cwd):
    candidates.append(cwd)

# 2) Walk upward in case Jupyter started in a child directory.
for parent in cwd.parents:
    if is_repo_root(parent):
        candidates.append(parent)

# 3) Fresh-reservation fallback: look for fraud-detection repos under /opt/app-root/src.
search_root = Path("/opt/app-root/src")
if search_root.exists():
    for candidate in search_root.glob("*/fraud-detection"):
        if is_repo_root(candidate.resolve()):
            candidates.append(candidate.resolve())

# De-duplicate.
unique_candidates = []
for candidate in candidates:
    if candidate not in unique_candidates:
        unique_candidates.append(candidate)

if not unique_candidates:
    raise FileNotFoundError(
        "Could not locate the fraud-detection repository. "
        "Clone the repo onto the workbench PVC, open this notebook from that repo, and rerun."
    )

if len(unique_candidates) > 1:
    raise RuntimeError(
        "Multiple fraud-detection repositories were found. "
        "To avoid writing the model to the wrong filesystem, remove the stale copy or "
        "open this notebook from the intended PVC-backed repo.\nFound:\n"
        + "\n".join(str(p) for p in unique_candidates)
    )

REPO_DIR = unique_candidates[0]
DATA_DIR = REPO_DIR / "data"
MODEL_V1_DIR = REPO_DIR / "models" / "fraud" / "1"
MODEL_V2_DIR = REPO_DIR / "models" / "fraud" / "2"
ARTIFACT_DIR = REPO_DIR / "artifact"

os.chdir(REPO_DIR)

MODEL_V2_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print("Repository:", REPO_DIR)
print("Data directory:", DATA_DIR)
print("v1 artifact:", MODEL_V1_DIR / "model.onnx")
print("v2 output:", MODEL_V2_DIR / "model.onnx")
print("v1 distance influence reduction:", f"{(1-DISTANCE_SENSITIVITY_MULTIPLIER)*100:.0f}%")

# Safety check: v1 should already exist from the clean setup flow.
v1_artifact = MODEL_V1_DIR / "model.onnx"
if not v1_artifact.is_file():
    raise FileNotFoundError(
        f"Expected v1 artifact was not found at {v1_artifact}. "
        "Run the v1 notebook successfully on the PVC before creating v2."
    )

print("v1 artifact verified:", v1_artifact, f"({v1_artifact.stat().st_size} bytes)")

In [ ]:
train_df = pd.read_csv(DATA_DIR / "train.csv")
validate_df = pd.read_csv(DATA_DIR / "validate.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")

target_candidates = ["fraud", "is_fraud", "label", "target", "Class"]
TARGET = next((c for c in target_candidates if c in train_df.columns), None)

if TARGET is None:
    raise ValueError(f"Could not identify target column. Columns: {list(train_df.columns)}")

missing_features = [c for c in FEATURES if c not in train_df.columns]
if missing_features:
    raise ValueError(f"Missing expected feature columns: {missing_features}")

X_train = train_df[FEATURES].astype("float32")
y_train = train_df[TARGET].astype(int)

X_validate = validate_df[FEATURES].astype("float32")
y_validate = validate_df[TARGET].astype(int)

X_test = test_df[FEATURES].astype("float32")
y_test = test_df[TARGET].astype(int)

print("Target:", TARGET)
print("Train:", X_train.shape)
print("Validation:", X_validate.shape)
print("Test:", X_test.shape)

## Recreate the deterministic fitted model in memory

This retrains the same deterministic model configuration used by v1.

For the demo:
- **v1** uses 15% of the fitted distance coefficient
- **v2** restores 100% of that fitted coefficient

The existing v1 ONNX artifact on the PVC is not overwritten.


In [ ]:
baseline_model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        solver="lbfgs",
        random_state=42,
    )),
])

baseline_model.fit(X_train, y_train)
print("Baseline model recreated in memory.")

In [ ]:
def evaluate(model, X, y, name):
    pred = model.predict(X)
    prob = model.predict_proba(X)[:, 1]
    metrics = {
        "accuracy": accuracy_score(y, pred),
        "precision": precision_score(y, pred, zero_division=0),
        "recall": recall_score(y, pred, zero_division=0),
        "f1": f1_score(y, pred, zero_division=0),
        "roc_auc": roc_auc_score(y, prob),
    }

    print(name)
    for k, v in metrics.items():
        print(f"  {k:10s}: {v:.4f}")

    return metrics


In [ ]:
v1_path = MODEL_V1_DIR / "model.onnx"

# Validate the already-existing ONNX file without rewriting it.
onnx.checker.check_model(onnx.load(v1_path))

print("Existing v1 artifact is valid:")
print(v1_path)
print("Size:", v1_path.stat().st_size, "bytes")

## Create v2: restore full distance sensitivity

v1 keeps only 15% of the fitted `distance_from_last_transaction` coefficient.
v2 restores the coefficient to its full fitted value, so the new model is more
sensitive to that fraud signal.


In [ ]:
# Recreate v1 in memory for an apples-to-apples comparison.
model_v1 = deepcopy(baseline_model)
distance_index = FEATURES.index(DISTANCE_FEATURE)
model_v1.named_steps["classifier"].coef_[0, distance_index] *= DISTANCE_SENSITIVITY_MULTIPLIER

# v2 is the full fitted model: restore 100% distance sensitivity.
model_v2 = deepcopy(baseline_model)

clf_v1 = model_v1.named_steps["classifier"]
clf_v2 = model_v2.named_steps["classifier"]

v1_coef = float(clf_v1.coef_[0, distance_index])
v2_coef = float(clf_v2.coef_[0, distance_index])

print("v1 distance coefficient:", v1_coef)
print("v2 distance coefficient:", v2_coef)
print("v2 restores the full fitted distance coefficient.")


In [ ]:
coef_comparison = pd.DataFrame({
    "feature": FEATURES,
    "v1_coefficient": clf_v1.coef_[0],
    "v2_coefficient": clf_v2.coef_[0],
})
coef_comparison["changed"] = ~np.isclose(
    coef_comparison["v1_coefficient"],
    coef_comparison["v2_coefficient"]
)
coef_comparison


In [ ]:
v1_metrics = evaluate(model_v1, X_test, y_test, "v1 — 85% less distance-sensitive")
v2_metrics = evaluate(model_v2, X_test, y_test, "v2 — full distance sensitivity")


## Find transactions where the increased sensitivity is most visible


In [ ]:
v1_probs = model_v1.predict_proba(X_test)[:, 1]
v2_probs = model_v2.predict_proba(X_test)[:, 1]

comparison = X_test.copy()
comparison["actual_label"] = y_test.to_numpy()
comparison["v1_fraud_probability"] = v1_probs
comparison["v2_fraud_probability"] = v2_probs
comparison["probability_increase"] = v2_probs - v1_probs

comparison.sort_values("probability_increase", ascending=False).head(10)


## Check the current Meridian suspicious preset

In [ ]:
meridian_suspicious = pd.DataFrame([{
    "distance_from_last_transaction": 950.0,
    "ratio_to_median_purchase_price": 12.0,
    "used_chip": 0.0,
    "used_pin_number": 0.0,
    "online_order": 1.0,
}], columns=FEATURES).astype("float32")

p1 = model_v1.predict_proba(meridian_suspicious)[0, 1]
p2 = model_v2.predict_proba(meridian_suspicious)[0, 1]

print(f"v1 fraud probability: {p1:.2%}")
print(f"v2 fraud probability: {p2:.2%}")
print(f"Increase: {(p2-p1):+.2%}")


## Export v2 to the PVC-backed repository

In [ ]:
initial_type = [("float_input", FloatTensorType([None, len(FEATURES)]))]

onnx_v2 = convert_sklearn(
    model_v2,
    initial_types=initial_type,
    options={id(model_v2.named_steps["classifier"]): {"zipmap": False}},
    target_opset=17,
)

v2_path = MODEL_V2_DIR / "model.onnx"
v2_path.write_bytes(onnx_v2.SerializeToString())

loaded_v2 = onnx.load(v2_path)
onnx.checker.check_model(loaded_v2)

print("Saved and validated v2:")
print(v2_path)
print("Size:", v2_path.stat().st_size, "bytes")

In [ ]:
metadata = {
    "model_version": 2,
    "repository": str(REPO_DIR),
    "features": FEATURES,
    "change_from_v1": {
        "feature": DISTANCE_FEATURE,
        "v1_sensitivity_multiplier": DISTANCE_SENSITIVITY_MULTIPLIER,
        "v2_sensitivity_multiplier": 1.0,
        "implementation": "v2 restores the full fitted distance logistic-regression coefficient",
    },
    "v1_artifact": str(v1_path),
    "v2_artifact": str(v2_path),
}

metadata_path = ARTIFACT_DIR / "model_v2_metadata.json"
metadata_path.write_text(json.dumps(metadata, indent=2))

print(metadata_path.read_text())


## Deploy v2 in OpenShift AI

Before editing the deployment, verify that this notebook printed a valid absolute path ending in:

```text
fraud-detection/models/fraud/2/model.onnx
```

Because the `fraud-detection` repository is stored directly on the workbench PVC, edit the existing
OpenShift AI deployment and change only the PVC-relative model path from:

```text
fraud-detection/models/fraud/1
```

to:

```text
fraud-detection/models/fraud/2
```

Keep this environment variable unchanged:

```text
MLSERVER_MODEL_URI=/mnt/models/model.onnx
```

Then wait for the `InferenceService` to become `READY=True`.

The Meridian UI and inference contract do not change — only the model artifact behind the endpoint changes.